In [ ]:
!python -m playwright install chromium

In [11]:
# threads_pipeline.py
# ============================================================
# THREADS POSTS SCRAPE & PARSE PIPELINE
# ============================================================
#
# PURPOSE:
#   Collect Threads post data for a list of brand accounts
#   and flatten it into a structured CSV for analysis.
#
# PIPELINE OVERVIEW:
#
#   STEP 0 — Load Accounts
#       Read user_id, username, and brand from
#       a user CSV (must contain: user_id, username, brand).
#
#   STEP 1 — Scrape (MODE = "scrape")
#       For each account, launch a Playwright browser using a
#       saved auth session (threads_state.json), navigate to
#       the profile page, intercept the GraphQL API request
#       (BarcelonaProfileThreadsTabRefetchable), inject the
#       pagination cursor for pages > 1, and save one raw
#       JSON file per page under:
#           Threads/posts_info/raw/<username>/
#       Errors are saved as separate *_ERROR.json files.
#       Each run appends a summary row to run_log.csv
#       (status, pages fetched, posts fetched, errors).
#
#   STEP 2 — Parse (MODE = "parse")
#       Read all saved raw JSON files recursively, skipping
#       error files. For each post, extract and flatten:
#         - Identity: brand, user ID, username, post ID
#         - Time: post timestamp, scrape timestamp
#         - Media: type (image/video/carousel), dimensions,
#                  cover URL, video URLs, audio flag
#         - Engagement: likes, comments, reshares, reposts,
#                        quotes
#         - Caption: full text, hashtags, mentions
#         - Flags: paid partnership, reply, pinned, counts
#                  disabled
#       Duplicate posts (by threads_post_id) are dropped.
#       Output: Threads/threads_posts_parsed.csv
#
# USAGE:
#   Set MODE = "scrape" to collect new data via browser.
#   Set MODE = "parse"  to process already-saved JSON files.
#   Both steps can be run independently.
#
# PREREQUISITES:
#   - threads_state.json   : saved Playwright auth session
#   - pip install playwright pandas
#   - playwright install chromium
#
# OUTPUT FILES:
#   Threads/posts_info/raw/<username>/*.json  — raw API pages
#   Threads/posts_info/run_log.csv            — scrape log
#   Threads/threads_posts_parsed.csv          — final table
# ============================================================

import asyncio
import json
import logging
import os
import re
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import parse_qs, urlencode
from ftfy import fix_text
import pandas as pd
from playwright.async_api import async_playwright
import random

In [28]:


# ======================
# Config
# ======================
STATE_FILE   = "threads_state.json"          # Playwright auth session
MAX_PAGES    = 3                            # Max pages to scrape per account

# --- Randomized delay ranges (seconds) ---
PAGE_DELAY_MIN      = 8       # between paginations (same account)
PAGE_DELAY_MAX      = 18
ACCOUNT_DELAY_MIN   = 30      # between accounts
ACCOUNT_DELAY_MAX   = 75

CURSOR_STATE_FILE = Path("Threads") / "post_info" / "cursor_state.json"


USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
]

VIEWPORTS = [
    {"width": 1280, "height": 800},
    {"width": 1440, "height": 900},
    {"width": 1920, "height": 1080},
    {"width": 1366, "height": 768},
]

RAW_DIR    = Path("Threads") / "post_info" / "raw"
PARSED_DIR = Path("Threads")
LOG_PATH   = Path("Threads") / "post_info" / "run_log.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PARSED_DIR.mkdir(parents=True, exist_ok=True)

In [21]:
# ======================
# Utilities  (shared with Instagram pipeline)
# ======================
def safe_get(d: Any, path: List[str], default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def safe_filename(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", "_", s)
    return re.sub(r"[^a-zA-Z0-9._-]", "", s)


def now_stamp() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def save_json(payload: Dict[str, Any], out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def load_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def append_log_row(row: Dict[str, Any], log_path: Path = LOG_PATH) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    df_row = pd.DataFrame([row])
    if log_path.exists():
        df_row.to_csv(log_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(log_path, mode="w", header=True, index=False)

In [22]:
# ======================
# CHANGE 3: Cursor persistence helpers
# Saves the last successful cursor per username so a crashed/interrupted
# run can resume from where it left off instead of starting over.
# ======================
def load_cursor_state() -> Dict[str, Any]:
    """Load the full cursor-state dict from disk. Returns {} if missing."""
    if CURSOR_STATE_FILE.exists():
        try:
            return load_json(CURSOR_STATE_FILE)
        except Exception:
            return {}
    return {}


def save_cursor_state(state: Dict[str, Any]) -> None:
    """Persist the cursor-state dict to disk."""
    CURSOR_STATE_FILE.parent.mkdir(parents=True, exist_ok=True)
    with CURSOR_STATE_FILE.open("w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


def get_saved_cursor(username: str) -> Optional[str]:
    """Return the last saved cursor for this username, or None."""
    state = load_cursor_state()
    entry = state.get(username, {})
    return entry.get("cursor") or None


def update_saved_cursor(username: str, cursor: str, page_num: int) -> None:
    """Persist the latest successful cursor for this username."""
    state = load_cursor_state()
    state[username] = {
        "cursor": cursor,
        "last_page": page_num,
        "updated_at": datetime.utcnow().isoformat(),
    }
    save_cursor_state(state)


def clear_saved_cursor(username: str) -> None:
    """Remove the cursor entry once an account is fully scraped."""
    state = load_cursor_state()
    state.pop(username, None)
    save_cursor_state(state)

In [23]:
# ======================
# Step 0: Read accounts from CSV
# ======================
def read_accounts_from_csv(csv_path: str) -> pd.DataFrame:
    """
    CSV must contain: user_id, username
    Recommended:      brand
    """
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    if "user_id" not in df.columns:
        raise ValueError("CSV must contain a 'user_id' column.")
    if "username" not in df.columns:
        raise ValueError("CSV must contain a 'username' column.")
    if "brand" not in df.columns:
        df["brand"] = ""

    df["user_id"]  = df["user_id"].astype(str).str.strip()
    df["username"] = df["username"].fillna("").astype(str).str.strip()
    df["brand"]    = df["brand"].fillna("").astype(str).str.strip()

    df = df[df["user_id"].str.len() > 0].copy()
    return df

In [24]:
# ======================
# Step 1A: Scrape one page via Playwright
# ======================
async def fetch_threads_page(
    username: str,
    user_id: str,
    cursor: Optional[str],
    page_num: int,
    pw_page,        # <-- Playwright Page object passed in (reused)
) -> Optional[Dict[str, Any]]:
    """
    Reuses an existing Playwright page object.
    Intercepts the GraphQL BarcelonaProfileThreadsTabRefetchable request,
    injects the cursor for pages > 1, and returns the response JSON.
    Returns None if no data was captured.
    """
    response_data = None

    async def handle_route(route, request):
        nonlocal cursor
        if "/graphql/query" not in request.url:
            await route.continue_()
            return

        post_data = request.post_data
        if not post_data or "BarcelonaProfileThreadsTabRefetchable" not in post_data:
            await route.continue_()
            return

        parsed = parse_qs(post_data)

        if cursor and page_num > 1:
            variables_str = parsed.get("variables", ["{}"])[0]
            variables = json.loads(variables_str)
            variables["after"] = cursor

            new_data = {}
            for key, value in parsed.items():
                if key == "variables":
                    new_data[key] = json.dumps(variables)
                else:
                    new_data[key] = value[0] if isinstance(value, list) else value

            await route.continue_(post_data=urlencode(new_data))
            logging.info("  Injected cursor: %s...", cursor[:40])
        else:
            await route.continue_()

    async def handle_response(response):
        nonlocal response_data
        try:
            if "/graphql/query" not in response.url:
                return
            req_data = response.request.post_data
            if req_data and "BarcelonaProfileThreadsTabRefetchable" in req_data:
                if not response_data:
                    body = await response.body()
                    response_data = json.loads(body.decode("utf-8"))
                    logging.info("  Captured GraphQL response (page %d)", page_num)
        except Exception as e:
            logging.warning("  Error capturing response: %s", e)

    # Register handlers on the reused page
    await pw_page.route("**/graphql/query", handle_route)
    pw_page.on("response", handle_response)

    await pw_page.goto(f"https://www.threads.com/@{username}")
    await pw_page.wait_for_load_state("networkidle")

    # Human-like scroll behaviour
    # Move mouse to a random position before scrolling
    await pw_page.mouse.move(
        random.randint(200, 900),
        random.randint(100, 500)
    )
    await asyncio.sleep(random.uniform(0.5, 1.5))
    
    # Then scroll naturally
    for _ in range(random.randint(3, 6)):
        scroll_amount = random.randint(300, 800)
        await pw_page.mouse.wheel(0, scroll_amount)
        await asyncio.sleep(random.uniform(0.8, 2.5))

    # CHANGE 2: randomized post-scroll pause (replaces fixed DELAY_BETWEEN_PAGES)
    await asyncio.sleep(random.uniform(PAGE_DELAY_MIN, PAGE_DELAY_MAX))

    # Unregister handlers so they don't fire on the next page load
    await pw_page.unroute("**/graphql/query", handle_route)
    pw_page.remove_listener("response", handle_response)

    return response_data


In [25]:

# ======================
# Step 1B: Scrape & save all pages for one account
# ======================
async def scrape_and_save_threads_jsons_for_account(
    username: str,
    user_id: str,
    brand: str = "",
    max_pages: int = MAX_PAGES,
    state_file: str = STATE_FILE,
    browser=None,           # <-- passed in from the outer loop (CHANGE 1)
) -> List[Path]:
    saved_paths: List[Path] = []
    username_clean = safe_filename(username) if username else f"user_{user_id}"
    stamp = int(datetime.utcnow().timestamp())

    account_raw_dir = RAW_DIR / username_clean
    account_raw_dir.mkdir(parents=True, exist_ok=True)

    # CHANGE 3: Try to resume from the last saved cursor
    saved_cursor = get_saved_cursor(username)
    if saved_cursor:
        logging.info("  [RESUME] Found saved cursor for %s, resuming from page after last checkpoint.", username)

    cursor    = saved_cursor   # None => start from the beginning
    page_num  = 1
    start_time = datetime.now()
    pages_fetched = 0
    posts_fetched = 0
    status = "success"
    error_type = ""
    error_message = ""

    # CHANGE 1: Create one context + one page per account; reuse across pages
    context  = await browser.new_context(
        storage_state=state_file,
        user_agent=random.choice(USER_AGENTS),
        viewport=random.choice(VIEWPORTS),
    )
    
    pw_page = await context.new_page()

    try:
        while page_num <= max_pages:
            logging.info("[%s] Scraping page %d...", username, page_num)

            try:
                response_data = await fetch_threads_page(
                    username=username,
                    user_id=user_id,
                    cursor=cursor,
                    page_num=page_num,
                    pw_page=pw_page,    # reused page (CHANGE 1)
                )
            except Exception as e:
                status = "failed"
                error_type = "playwright_error"
                error_message = str(e)

                err_path = account_raw_dir / f"{username_clean}_{stamp}_page{page_num}_ERROR.json"
                save_json(
                    {
                        "username": username,
                        "user_id": user_id,
                        "brand": brand,
                        "page": page_num,
                        "error_type": error_type,
                        "error": error_message,
                    },
                    err_path,
                )
                saved_paths.append(err_path)
                break

            if not response_data:
                logging.warning("  No data captured for page %d, stopping.", page_num)
                status = "failed" if pages_fetched == 0 else "partial"
                error_type = "no_data"
                break

            response_data["_meta"] = {
                "username": username,
                "user_id": user_id,
                "brand": brand,
                "page": page_num,
                "scrape_ts": stamp,
                "cursor_used": cursor or "",
            }

            out_path = account_raw_dir / f"{username_clean}_{stamp}_page{page_num}.json"
            save_json(response_data, out_path)
            saved_paths.append(out_path)
            logging.info("  Saved -> %s", out_path.name)

            edges = safe_get(response_data, ["data", "mediaData", "edges"], []) or []
            page_posts = sum(
                len(edge.get("node", {}).get("thread_items", []))
                for edge in edges
                if isinstance(edge, dict)
            )
            pages_fetched += 1
            posts_fetched += page_posts

            # Pagination + cursor persistence
            try:
                page_info  = safe_get(response_data, ["data", "mediaData", "page_info"], {}) or {}
                has_next   = page_info.get("has_next_page", False)
                new_cursor = page_info.get("end_cursor") or page_info.get("cursor") or ""

                if not has_next or not new_cursor:
                    logging.info("  Reached end of posts for %s.", username)
                    clear_saved_cursor(username)   # CHANGE 3: clean up on natural finish
                    break

                cursor = new_cursor
                # CHANGE 3: persist cursor immediately after each successful page
                update_saved_cursor(username, cursor, page_num)
                logging.info("  Cursor saved for %s (page %d).", username, page_num)

            except Exception as e:
                status = "partial"
                error_type = "pagination_error"
                error_message = str(e)
                break

            page_num += 1

    finally:
        # CHANGE 1: Always close the context (not the browser) when done with this account
        await pw_page.close()
        await context.close()
        logging.info("  Context closed for %s.", username)

    end_time = datetime.now()
    if status == "failed" and pages_fetched > 0:
        status = "partial"

    append_log_row(
        {
            "run_id":        stamp,
            "username":      username,
            "user_id":       user_id,
            "brand":         brand,
            "started_at":    start_time.isoformat(),
            "ended_at":      end_time.isoformat(),
            "status":        status,
            "pages_fetched": pages_fetched,
            "posts_fetched": posts_fetched,
            "error_type":    error_type,
            "error_message": error_message,
            "raw_files":     ";".join([p.name for p in saved_paths]),
        },
        log_path=LOG_PATH,
    )

    return saved_paths

In [26]:
# ======================
# Step 1C: Loop over all accounts in CSV
# ======================
async def scrape_threads_from_csv(
    csv_path: str,
    max_pages: int = MAX_PAGES,
    state_file: str = STATE_FILE,
) -> None:
    accounts = read_accounts_from_csv(csv_path)
    total = len(accounts)

    # CHANGE 1: Single browser for the entire run
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        logging.info("Browser launched. Will be reused for all %d accounts.", total)

        try:
            for i, row in accounts.iterrows():
                username = str(row["username"])
                user_id  = str(row["user_id"])
                brand    = str(row.get("brand", "") or "")

                logging.info(
                    "[%d/%d] username=%s  user_id=%s  brand=%s",
                    i + 1, total, username, user_id, brand,
                )

                saved = await scrape_and_save_threads_jsons_for_account(
                    username=username,
                    user_id=user_id,
                    brand=brand,
                    max_pages=max_pages,
                    state_file=state_file,
                    browser=browser,    # CHANGE 1: pass shared browser
                )
                logging.info("  -> saved %d file(s)", len(saved))

                # CHANGE 2: randomized inter-account delay (skip after last account)
                if i + 1 < total:
                    wait_sec = random.uniform(ACCOUNT_DELAY_MIN, ACCOUNT_DELAY_MAX)
                    logging.info("  Waiting %.1fs before next account...", wait_sec)
                    await asyncio.sleep(wait_sec)

        finally:
            # CHANGE 1: Browser closes only once, after everything is done
            await browser.close()
            logging.info("Browser closed.")



In [27]:

# ======================
# MAIN — STEP 1: SCRAPE
# Run this block to collect raw JSON pages from Threads.
# Reads accounts from CSV, launches Playwright, saves one
# JSON file per page under Threads/posts_info/raw/<username>/
# ======================
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

await scrape_threads_from_csv(
    csv_path="Threads/threads_user_info.csv",   # must have: user_id, username, brand
    max_pages=MAX_PAGES,
    state_file=STATE_FILE,
)

INFO: Browser launched. Will be reused for all 1 accounts.
INFO: [1/1] username=nike  user_id=13460080  brand=
INFO: [nike] Scraping page 1...
INFO:   Captured GraphQL response (page 1)
INFO:   Saved -> nike_1773106708_page1.json
INFO:   Cursor saved for nike (page 1).
INFO: [nike] Scraping page 2...
INFO:   Injected cursor: QVFCWGJBMGQ4NlBBdTBzbVJObG93Y3FhdmdwNG9f...
INFO:   Captured GraphQL response (page 2)
INFO:   Saved -> nike_1773106708_page2.json
INFO:   Cursor saved for nike (page 2).
INFO: [nike] Scraping page 3...
INFO:   Injected cursor: QVFCQk5ITGRyaF9qdnBoYXJQX2ZqYkJaUlpTMDdy...
INFO:   Captured GraphQL response (page 3)
INFO:   Saved -> nike_1773106708_page3.json
INFO:   Cursor saved for nike (page 3).
INFO:   Context closed for nike.
INFO:   -> saved 3 file(s)
INFO: Browser closed.


In [15]:
# ======================
# Step 2 helpers: image extraction
# ======================
HASHTAG_RE = re.compile(r"#(\w+)")
MENTION_RE = re.compile(r"@(\w+)")
def fix_mojibake(text: str) -> str:
    return fix_text(text) if text else ""

def extract_hashtags(text: str) -> str:
    return ", ".join(HASHTAG_RE.findall(text)) if text else ""


def extract_mentions(text: str) -> str:
    return ", ".join(MENTION_RE.findall(text)) if text else ""


def best_image_candidate(obj: Dict[str, Any]) -> Tuple[str, Optional[int], Optional[int]]:
    candidates = safe_get(obj, ["image_versions2", "candidates"], []) or []
    if not isinstance(candidates, list) or not candidates:
        return "", None, None

    best, best_area = None, -1
    for c in candidates:
        if not isinstance(c, dict):
            continue
        url = c.get("url")
        w, h = c.get("width"), c.get("height")
        if not url:
            continue
        area = (w * h) if (isinstance(w, int) and isinstance(h, int) and w > 0 and h > 0) else 0
        if area > best_area:
            best, best_area = c, area

    if not best:
        first = candidates[0]
        return first.get("url", "") or "", first.get("width"), first.get("height")
    return best.get("url", "") or "", best.get("width"), best.get("height")


# ======================
# Step 2A: Parse one raw JSON payload into post rows
# ======================
def extract_posts_from_payload(
    raw_payload: Dict[str, Any],
    brand: str = "",
) -> List[Dict[str, Any]]:
    """
    Flatten one page payload (the API response dict) into a list of post rows.
    """
    rows: List[Dict[str, Any]] = []

    edges = safe_get(raw_payload, ["mediaData", "edges"], []) or []
    if not isinstance(edges, list):
        return rows

    for edge in edges:
        if not isinstance(edge, dict):
            continue

        node = edge.get("node", {}) or {}
        thread_items = node.get("thread_items", []) or []

        for item in thread_items:
            if not isinstance(item, dict):
                continue

            post = item.get("post", {}) or {}
            if not isinstance(post, dict):
                continue

            # --- Identity ---
            user           = post.get("user", {}) or {}
            user_id        = user.get("id") or user.get("pk") or ""
            username       = user.get("username", "") or ""
            full_name      = user.get("full_name", "") or ""
            is_verified    = bool(user.get("is_verified", False))

            # --- Post IDs + Timestamp ---
            threads_post_id = str(post.get("id", "") or "")
            threads_pk      = str(post.get("pk", "") or "")
            taken_at = post.get("taken_at", None)
            timestamp_unix = int(taken_at) if isinstance(taken_at, (int, float)) and taken_at else None

            # --- Caption ---
            caption_text = fix_mojibake(safe_get(post, ["caption", "text"], "") or "")
            hashtags     = extract_hashtags(caption_text)
            mentions     = extract_mentions(caption_text)

            # --- Flags ---
            is_paid_partnership          = bool(post.get("is_paid_partnership", False))
            like_and_view_counts_disabled = bool(post.get("like_and_view_counts_disabled", False))
            has_audio                    = post.get("has_audio") or ""
            media_type                   = post.get("media_type") or ""

            tpai = post.get("text_post_app_info") or {}
            is_reply  = tpai.get("is_reply", False)
            is_pinned = (tpai.get("pinned_post_info", {}) or {}).get("is_pinned_to_profile", False)

            # --- Engagement ---
            like_count         = post.get("like_count", 0) or 0
            reshare_count      = tpai.get("reshare_count", 0) or 0
            direct_reply_count = tpai.get("direct_reply_count", 0) or 0
            repost_count       = tpai.get("repost_count", 0) or 0
            quote_count        = tpai.get("quote_count", 0) or 0

            original_width  = post.get("original_width")
            original_height = post.get("original_height")

            # --- Media ---
            carousel_media     = post.get("carousel_media", []) or []
            image_count        = 0
            carousel_media_urls: List[str] = []
            cover_url, cover_width, cover_height = "", None, None

            if carousel_media and isinstance(carousel_media, list) and media_type != 2:
                image_count = len(carousel_media)
                cover_url, cover_width, cover_height = best_image_candidate(carousel_media[0])
                for m in carousel_media:
                    if isinstance(m, dict):
                        u, _, _ = best_image_candidate(m)
                        if u:
                            carousel_media_urls.append(u)
            elif media_type != 2:
                cover_url, cover_width, cover_height = best_image_candidate(post)
                image_count = 1 if cover_url else 0
                if cover_url:
                    carousel_media_urls.append(cover_url)
            else:
                cover_url, cover_width, cover_height = best_image_candidate(post)

            video_versions = post.get("video_versions") or []
            video_urls     = {v.get("type"): v.get("url") for v in video_versions}
            hd_url = video_urls.get(103, "")
            sd_url = video_urls.get(101, "")
            
            
            rows.append({
                # --- Identity ---
                "brand":           brand,
                "user_id":         user_id,
                "username":        username,
                "full_name":       full_name,
                "is_verified":     is_verified,

                # --- Post IDs + Time ---
                "threads_post_id": threads_post_id,
                "threads_pk":      threads_pk,
                "timestamp":       timestamp_unix,

                # --- Flags ---
                "is_paid_partnership":           is_paid_partnership,
                "is_reply":                      is_reply,
                "like_and_view_counts_disabled": like_and_view_counts_disabled,
                "is_pinned":                     is_pinned,

                # --- Caption ---
                "caption_text": caption_text,
                "hashtags":     hashtags,
                "mentions":     mentions,

                # --- Engagement ---
                "like_count":    like_count,
                "reshare_count": reshare_count,
                "comment_count": direct_reply_count,
                "repost_count":  repost_count,
                "quote_count":   quote_count,

                # --- Media ---
                "media_type":          media_type,
                "image_count":         image_count,
                "has_audio":           has_audio,
                "original_width":      original_width,
                "original_height":     original_height,
                "cover_url":           cover_url,
                "cover_width":         cover_width,
                "cover_height":        cover_height,
                "carousel_media_urls": ", ".join(carousel_media_urls),
                "video_hd_url":        hd_url,
                "video_sd_url":        sd_url,
            })

    return rows


# ======================
# Step 2B: Parse all saved JSONs -> CSV
# ======================
def parse_saved_threads_jsons(
    raw_dir: Path = RAW_DIR,
    out_csv: str  = str(PARSED_DIR / "threads_posts_parsed.csv"),
) -> pd.DataFrame:
    """
    Recursively reads ALL saved JSON files under:
        RAW_DIR / <username> / *.json

    Excludes *ERROR.json files, parses posts into one table,
    and writes a single CSV.
    """
    json_files = sorted(
        [p for p in raw_dir.rglob("*.json") if "ERROR" not in p.name]
    )

    all_rows: List[Dict[str, Any]] = []

    for p in json_files:
        try:
            wrapper = load_json(p)
        except Exception as e:
            logging.warning("Failed to load %s: %s", p, e)
            continue

        meta  = wrapper.get("_meta", {}) or {}
        brand = meta.get("brand", "")
        page  = meta.get("page", "")
        stamp = meta.get("scrape_ts", "")

        # The actual API response is under wrapper["data"]
        api_payload = wrapper.get("data", {}) or {}

        rows = extract_posts_from_payload(api_payload, brand=brand)

        for r in rows:
            r["scrape_ts"]   = stamp
            r["page"]        = page
            r["_raw_file"]   = p.name
            r["_username"]   = p.parent.name   # folder name = username

        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)

    # Deduplicate at post level (pagination overlap protection)
    if not df.empty and "threads_post_id" in df.columns:
        df = (
            df
            .drop_duplicates(subset=["threads_post_id"], keep="first")
            .reset_index(drop=True)
        )

    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False, encoding="utf-8")

    logging.info(
        "[DONE] Parsed %d JSON file(s) -> rows=%d -> %s",
        len(json_files), len(df), out_csv,
    )
    return df

In [16]:
# ======================
# MAIN — STEP 2: PARSE
# Run this block (independently) to convert saved raw JSONs
# into a flat CSV at Threads/threads_posts_parsed.csv
# ======================
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
#
    df_posts = parse_saved_threads_jsons(
        raw_dir=RAW_DIR,
        out_csv=str(PARSED_DIR / "threads_posts_parsed.csv"),
    )


INFO: [DONE] Parsed 15 JSON file(s) -> rows=149 -> Threads/threads_posts_parsed.csv


In [17]:
df_posts

,brand,user_id,username,full_name,is_verified,threads_post_id,threads_pk,timestamp,is_paid_partnership,is_reply,...,cover_url,cover_width,cover_height,carousel_media_urls,video_hd_url,video_sd_url,scrape_ts,page,_raw_file,_username
0,,63454848350,nike,Nike,True,3836254256739690476_63454848350,3836254256739690476,1771537166,False,False,...,https://scontent-yyz1-1.cdninstagram.com/v/t51...,1440.0,1800.0,https://scontent-yyz1-1.cdninstagram.com/v/t51...,,,1773102892,1,nike_1773102892_page1.json,nike
1,,63454848350,nike,Nike,True,3835523584857660664_63454848350,3835523584857660664,1771450063,False,False,...,https://scontent-yyz1-1.cdninstagram.com/v/t51...,1440.0,1801.0,https://scontent-yyz1-1.cdninstagram.com/v/t51...,,,1773102892,1,nike_1773102892_page1.json,nike
2,,63454848350,nike,Nike,True,3829507873022197862_63454848350,3829507873022197862,1770732934,False,False,...,https://scontent-yyz1-1.cdninstagram.com/v/t51...,1440.0,1800.0,https://scontent-yyz1-1.cdninstagram.com/v/t51...,,,1773102892,1,nike_1773102892_page1.json,nike
3,,63454848350,nike,Nike,True,3828928273242473482_63454848350,3828928273242473482,1770663840,False,False,...,https://scontent-yyz1-1.cdninstagram.com/v/t51...,1440.0,1800.0,https://scontent-yyz1-1.cdninstagram.com/v/t51...,,,1773102892,1,nike_1773102892_page1.json,nike
4,,63454848350,nike,Nike,True,3828458202728602651_63454848350,3828458202728602651,1770607822,False,False,...,https://scontent-yyz1-1.cdninstagram.com/v/t51...,640.0,800.0,,https://scontent-yyz1-1.cdninstagram.com/o1/v/...,https://scontent-yyz1-1.cdninstagram.com/o1/v/...,1773102892,1,nike_1773102892_page1.json,nike
5,,63454848350,nike,Nike,True,3828284063153436018_63454848350,3828284063153436018,1770587045,False,False,...,https://scontent-yyz1-1.cdninstagram.com/v/t51...,1440.0,1800.0,https://scontent-yyz1-1.cdninstagram.com/v/t51...,,,1773102892,1,nike_1773102892_page1.json,nike
6,,63454848350,nike,Nike,True,3823734228824199950_63454848350,3823734228824199950,1770044712,False,False,...,https://scontent-yyz1-1.cdninstagram.com/v/t51...,640.0,1136.0,,https://scontent-yyz1-1.cdninstagram.com/o1/v/...,https://scontent-yyz1-1.cdninstagram.com/o1/v/...,1773102892,1,nike_1773102892_page1.json,nike
7,,63454848350,nike,Nike,True,3823737271506502434_63454848350,3823737271506502434,1770045025,False,True,...,,NaN,NaN,,,,1773102892,1,nike_1773102892_page1.json,nike
8,,63454848350,nike,Nike,True,3749745908608258374_63454848350,3749745908608258374,1761224585,False,False,...,https://scontent-yyz1-1.cdninstagram.com/v/t51...,640.0,800.0,,https://scontent-yyz1-1.cdninstagram.com/o1/v/...,https://scontent-yyz1-1.cdninstagram.com/o1/v/...,1773102892,1,nike_1773102892_page1.json,nike
9,,63454848350,nike,Nike,True,3720846444036471382_63454848350,3720846444036471382,1757779482,False,False,...,https://scontent-yyz1-1.cdninstagram.com/v/t51...,640.0,800.0,https://scontent-yyz1-1.cdninstagram.com/v/t51...,,,1773102892,1,nike_1773102892_page1.json,nike
